In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import sys
import csv
df = pd.read_csv("/content/drive/MyDrive/Phase_II Project(Next Word)/hindi_cleaned_final (1).csv")
df.head()

,text
0,एक विराम चिह्न है जिसे विस्मयादिबोधक चिह्न कहत...
1,पिक्चर्स या एंड पिक्चर्स एक टेलीविजन चैनल है ज...
2,जे॰बी॰ से ने अपनी पुस्तक ट्रेट डी एकनोमिक पोल्...
3,अक्कड़ के राजा नराम सिन के शासनकाल से संबंधित ...
4,त ख्वाबनि जो छा थींदो अनुवाद सपनों की बुवाई कर...


In [ ]:
with open("hindi_corpus.txt", "w", encoding="utf-8") as f:
    for line in df["text"]:
        f.write(str(line) + "\n")

In [ ]:
with open("hindi_corpus.txt", "w", encoding="utf-8") as f:
    for line in df["text"]:
        f.write(str(line) + "\n")

In [ ]:
df.head()

,text
0,एक विराम चिह्न है जिसे विस्मयादिबोधक चिह्न कहत...
1,पिक्चर्स या एंड पिक्चर्स एक टेलीविजन चैनल है ज...
2,जे॰बी॰ से ने अपनी पुस्तक ट्रेट डी एकनोमिक पोल्...
3,अक्कड़ के राजा नराम सिन के शासनकाल से संबंधित ...
4,त ख्वाबनि जो छा थींदो अनुवाद सपनों की बुवाई कर...


In [ ]:
from datasets import load_dataset

dataset = load_dataset("text", data_files={"train": "hindi_corpus.txt"})

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
import math
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model

In [ ]:
dataset = load_dataset("text", data_files={"train": "hindi_corpus.txt"})
# Train-test split (5%)
dataset = dataset["train"].train_test_split(test_size=0.05)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/156788 [00:00<?, ? examples/s]

Map:   0%|          | 0/8253 [00:00<?, ? examples/s]

## Model 1 : **EleutherAI/gpt-neo-125M**

In [ ]:
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-125M")

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

In [ ]:
!pip install --upgrade torchao
# APPLY LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 8.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
training_args = TrainingArguments(
    output_dir="./neo_results",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_dir="./logs",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.989273,0.987392
2,0.940848,0.942479
3,0.931294,0.930687


TrainOutput(global_step=117591, training_loss=0.9909789923834253, metrics={'train_runtime': 8807.6904, 'train_samples_per_second': 53.404, 'train_steps_per_second': 13.351, 'total_flos': 3.082213766253773e+16, 'train_loss': 0.9909789923834253, 'epoch': 3.0})

In [ ]:
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])

print(f"GPT-Neo Perplexity: {perplexity}")

GPT-Neo Perplexity: 2.536250102724423


In [ ]:
model.save_pretrained("./neo_model")
tokenizer.save_pretrained("./neo_model")

('./neo_model/tokenizer_config.json', './neo_model/tokenizer.json')

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./neo_model",
    tokenizer=tokenizer
)

output = generator(
    "भारत एक",
    max_length=20,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

print("Output:", output[0]["generated_text"])

Output: महान राष्ट्र है


## Model 2: **TinyLlama 1.1B**

In [ ]:
# Install 4-bit quantization
!pip install -U bitsandbytes>=0.46.1



In [ ]:
import math
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model


In [ ]:
dataset = load_dataset("text", data_files={"train": "hindi_corpus.txt"})

dataset = dataset["train"].train_test_split(test_size=0.05)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/156788 [00:00<?, ? examples/s]

Map:   0%|          | 0/8253 [00:00<?, ? examples/s]

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemError("CUDA is not available. Please ensure you are using a GPU runtime.")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16"
)

model = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    quantization_config=bnb_config,
    device_map="auto"
)

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama_results",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    logging_dir="./logs",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])

print(f"TinyLlama Perplexity: {perplexity}")

In [ ]:

model.save_pretrained("./tinyllama_model")
tokenizer.save_pretrained("./tinyllama_model")

In [ ]:
# ===============================
# STEP 10: INFERENCE
# ===============================
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./tinyllama_model",
    tokenizer=tokenizer
)

output = generator(
    "भारत एक",
    max_length=20,
    do_sample=True,
    top_k=50,
    top_p=0.95
)

print("Output:", output[0]["generated_text"])